# Внимание и Трансформер (Attention & Transformer) - как нейросети "читают"

**Роль:** Преподаватель по ML
**Уровень:** средний (основы PyTorch + RNN/LSTM)
**Время прохождения:** ~90-120 минут

---

### Чему вы научитесь

После прохождения этого ноутбука вы:
- поймёте, **зачем** нужен механизм внимания и какие проблемы он решает;
- реализуете **Self-Attention** с нуля на NumPy - шаг за шагом;
- визуализируете и **интерактивно** исследуете внимание;
- разберёте **Multi-Head Attention** и поймёте, зачем нужны несколько "голов";
- поймёте **позиционное кодирование** и почему без него трансформер не работает;
- соберёте полный **Transformer Encoder** из блоков;
- **обучите** модель для классификации текста;
- увидите, **на что** модель обращает внимание (heatmap);
- **сравните** RNN, LSTM и Transformer на одной задаче.

### Принцип этого блокнота

> Вы - автор, не пользователь. Каждая строка видна. Каждое действие можно сломать и починить. Никаких "чёрных ящиков".


## Шаг 1. Настройка окружения

Подключаем все библиотеки, которые понадобятся в ноутбуке.
Ничего необычного - PyTorch, NumPy, Matplotlib и виджеты для интерактивности.


In [ ]:
# Импортируем PyTorch - основной фреймворк для нейросетей
import torch
# Импортируем модуль нейросетей из PyTorch
import torch.nn as nn
# Импортируем модуль оптимизаторов
import torch.optim as optim
# Импортируем функциональные операции (softmax, relu и т.д.)
import torch.nn.functional as F
# Импортируем утилиты для работы с данными
from torch.utils.data import DataLoader, Dataset

# Импортируем NumPy для вычислений с нуля
import numpy as np
# Импортируем Matplotlib для визуализаций
import matplotlib.pyplot as plt
# Импортируем цветовые карты для heatmap
from matplotlib.colors import LinearSegmentedColormap
# Подключаем интерактивные виджеты
import ipywidgets as widgets
# Для отображения виджетов
from IPython.display import display, HTML

# Для генерации случайных данных
import random
# Для копирования объектов
import copy
# Для работы с путями
import os

# Настройка размера графиков
plt.rcParams['figure.figsize'] = (10, 6)
# Настройка качества графиков
plt.rcParams['figure.dpi'] = 100
# Включаем поддержку кириллицы в графиках
plt.rcParams['font.size'] = 12

# Фиксируем случайность для воспроизводимости
torch.manual_seed(42)
# Фиксируем случайность NumPy
np.random.seed(42)
# Фиксируем случайность Python
random.seed(42)

# Проверяем доступность GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Выводим информацию об устройстве
print(f'PyTorch version: {torch.__version__}')
# Выводим устройство вычислений
print(f'Device: {device}')
# Выводим версию NumPy
print(f'NumPy version: {np.__version__}')


## Шаг 2. Почему нужно Внимание? Проблема RNN

### Проблема последовательной обработки

RNN (и LSTM) обрабатывают последовательность **по одному элементу за раз**:

```
x1 -> h1 -> h2 -> h3 -> ... -> hT
```

Это создаёт три проблемы:

1. **Дальний контекст** - информация из начала последовательности "размывается" к концу,
   даже в LSTM. Связь между словом 1 и словом 100 - очень слабая.

2. **Последовательность = медленность** - нельзя распараллелить обработку.
   Шаг t+1 ждёт результата шага t. Для длинных текстов - это долго.

3. **Фиксированное "сжатие"** - всё предложение сжимается в один вектор h_T.
   Информация теряется неизбежно.

### Идея Внимания (Attention)

Вместо того чтобы "сжимать" всё в один вектор, пусть **каждый элемент**
последовательности может **посмотреть на любые другие элементы** напрямую!

```
Вместо:  x1 -> h1 -> h2 -> h3   (последовательно)
Делаем:  x1, x2, x3 -> Attention -> y1, y2, y3   (параллельно!)
```

Каждый выход y_i - это **взвешенная сумма** всех входов,
где веса определяют, "на что обратить внимание".


In [ ]:
# Визуализация: RNN (последовательно) vs Attention (параллельно)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Левый график: RNN - последовательная обработка ---
ax = axes[0]
ax.set_title('RNN: последовательная обработка', fontsize=14, fontweight='bold')
# Рисуем узлы RNN
for i in range(5):
    # Рисуем круг - скрытое состояние
    circle = plt.Circle((i * 2, 0.5), 0.4, fill=True, color='#FF6B6B', alpha=0.8)
    ax.add_patch(circle)
    # Подписываем скрытое состояние
    ax.text(i * 2, 0.5, f'h{i+1}', ha='center', va='center', fontsize=12, color='white', fontweight='bold')
    # Рисуем входной узел
    rect = plt.Rectangle((i * 2 - 0.3, -0.8), 0.6, 0.4, fill=True, color='#4ECDC4', alpha=0.8)
    ax.add_patch(rect)
    # Подписываем вход
    ax.text(i * 2, -0.6, f'x{i+1}', ha='center', va='center', fontsize=10, color='white')
    # Стрелка от входа к скрытому состоянию
    ax.annotate('', xy=(i * 2, 0.1), xytext=(i * 2, -0.4), arrowprops=dict(arrowstyle='->', color='#333'))
    # Стрелка между скрытыми состояниями (последовательная связь)
    if i < 4:
        ax.annotate('', xy=((i+1) * 2 - 0.4, 0.5), xytext=(i * 2 + 0.4, 0.5), arrowprops=dict(arrowstyle='->', color='#FF6B6B', lw=2))

# Добавляем подпись проблемы
ax.text(4, -1.4, 'Каждый шаг ждёт предыдущий!', ha='center', fontsize=11, color='#FF6B6B', style='italic')
ax.set_xlim(-1, 9)
ax.set_ylim(-1.8, 1.2)
ax.axis('off')

# --- Правый график: Attention - параллельная обработка ---
ax = axes[1]
ax.set_title('Attention: параллельная обработка', fontsize=14, fontweight='bold')
# Рисуем входные узлы (нижний ряд)
for i in range(5):
    rect = plt.Rectangle((i * 2 - 0.3, -0.3), 0.6, 0.4, fill=True, color='#4ECDC4', alpha=0.8)
    ax.add_patch(rect)
    ax.text(i * 2, -0.1, f'x{i+1}', ha='center', va='center', fontsize=10, color='white')

# Рисуем выходные узлы (верхний ряд)
for i in range(5):
    circle = plt.Circle((i * 2, 1.5), 0.4, fill=True, color='#45B7D1', alpha=0.8)
    ax.add_patch(circle)
    ax.text(i * 2, 1.5, f'y{i+1}', ha='center', va='center', fontsize=12, color='white', fontweight='bold')

# Рисуем связи внимания - каждый выход связан со всеми входами
# Используем разные прозрачности для разных "весов внимания"
for i in range(5):
    for j in range(5):
        # Вес внимания - больше для близких элементов
        weight = 1.0 / (abs(i - j) + 1)
        ax.annotate('', xy=(i * 2, 1.1), xytext=(j * 2, 0.1),
                     arrowprops=dict(arrowstyle='->', color='#45B7D1', alpha=weight * 0.6, lw=weight * 2))

# Добавляем подпись
ax.text(4, -0.9, 'Все связи параллельно!', ha='center', fontsize=11, color='#45B7D1', style='italic')
ax.set_xlim(-1, 9)
ax.set_ylim(-1.3, 2.2)
ax.axis('off')

plt.tight_layout()
plt.show()
# Выводим пояснение
print('Слева: RNN обрабатывает последовательно - шаг за шагом')
print('Справа: Attention связывает все элементы сразу - параллельно')


## Шаг 3. Self-Attention: основная идея

### Query, Key, Value - три компонента внимания

Механизм внимания вдохновлён **поисковыми системами**:

- **Query (Q)** - "что я ищу?" (запрос)
- **Key (K)** - "что здесь есть?" (ключ, описание элемента)
- **Value (V)** - "что я получу" (содержимое элемента)

### Аналогия с библиотекой

Представьте, что вы в библиотеке:
1. У вас есть **запрос** (Query): "книги про котов"
2. У каждой книги есть **ключ** (Key): название, теги, аннотация
3. У каждой книги есть **содержимое** (Value): сам текст книги

Вы **сравниваете** запрос с ключами -> получаете "оценки релевантности"
Берёте **взвешенную сумму** содержимого по этим оценкам -> результат!

### Формула Self-Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Где:
- $d_k$ - размерность ключа (для стабильности градиентов)
- softmax превращает оценки в вероятности (сумма = 1)


## Шаг 4. Self-Attention с нуля на NumPy

Реализуем механизм внимания шаг за шагом, чтобы увидеть каждый этап.
Используем простой пример с 4 словами, каждое представлено вектором из 3 чисел.


In [ ]:
# --- Шаг 4.1: Создаём входные данные ---
# Представим предложение из 4 слов
# Каждое слово - вектор размерности 3 (в реальности 512+)
np.random.seed(42)  # Фиксируем случайность
# Матрица входов: 4 слова x 3 признака
X = np.random.randn(4, 3)
# Выводим входную матрицу
print('Входная матрица X (4 слова x 3 признака):')
# Печатаем с округлением
print(np.round(X, 3))
# Размерность входа
print(f'\nФорма X: {X.shape}')


In [ ]:
# --- Шаг 4.2: Создаём матрицы весов для Q, K, V ---
# W_q: преобразует вход в Query (что я ищу)
W_q = np.random.randn(3, 3) * 0.5  # Инициализация с маленькими значениями
# W_k: преобразует вход в Key (что я содержу)
W_k = np.random.randn(3, 3) * 0.5
# W_v: преобразует вход в Value (что я передам)
W_v = np.random.randn(3, 3) * 0.5

# Вычисляем Query: X @ W_q
Q = X @ W_q  # Матричное умножение: (4,3) @ (3,3) = (4,3)
# Вычисляем Key: X @ W_k
K = X @ W_k  # (4,3) @ (3,3) = (4,3)
# Вычисляем Value: X @ W_v
V = X @ W_v  # (4,3) @ (3,3) = (4,3)

# Выводим результаты
print('Query (Q) - "что я ищу":')
print(np.round(Q, 3))  # Печатаем с округлением
print(f'\nKey (K) - "что я содержу":')
print(np.round(K, 3))  # Печатаем с округлением
print(f'\nValue (V) - "что я передам":')
print(np.round(V, 3))  # Печатаем с округлением


In [ ]:
# --- Шаг 4.3: Вычисляем оценки внимания (attention scores) ---
# Оценка = Q @ K^T (скалярное произведение запроса и ключа)
# K.T - транспонированная матрица ключей
scores = Q @ K.T  # (4,3) @ (3,4) = (4,4)
# Выводим оценки до масштабирования
print('Оценки внимания (Q @ K^T):')
print(np.round(scores, 3))
# Каждая строка - внимание одного слова ко всем остальным
# Проблема: значения могут быть слишком большими -> softmax "насыщается"
print(f'\nМинимальная оценка: {scores.min():.3f}')
print(f'Максимальная оценка: {scores.max():.3f}')


In [ ]:
# --- Шаг 4.4: Масштабирование (divide by sqrt(d_k)) ---
# d_k - размерность ключа (количество признаков)
d_k = K.shape[1]  # = 3
# Масштабируем оценки: делим на корень из d_k
scaled_scores = scores / np.sqrt(d_k)  # Стабилизация градиентов
# Выводим масштабированные оценки
print(f'd_k = {d_k}, sqrt(d_k) = {np.sqrt(d_k):.3f}')
print(f'\nМасштабированные оценки (scores / sqrt(d_k)):')
print(np.round(scaled_scores, 3))
# Масштабирование предотвращает слишком большие значения в softmax
print(f'\nПосле масштабирования: min={scaled_scores.min():.3f}, max={scaled_scores.max():.3f}')


In [ ]:
# --- Шаг 4.5: Применяем softmax ---
# Softmax превращает оценки в вероятности (0..1, сумма=1)
def softmax(x):
    # Вычитаем max для численной стабильности
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    # Делим на сумму экспонент
    return e_x / e_x.sum(axis=-1, keepdims=True)

# Применяем softmax к каждой строке (внимание каждого слова)
attention_weights = softmax(scaled_scores)  # (4,4) матрица вероятностей
# Выводим веса внимания
print('Веса внимания (после softmax):')
print(np.round(attention_weights, 3))
# Проверяем, что сумма каждой строки = 1
print(f'\nСумма по строкам (должна быть 1.0): {np.round(attention_weights.sum(axis=1), 6)}')
# Каждая строка показывает, какое слово на какие другие "смотрит"


In [ ]:
# --- Шаг 4.6: Вычисляем выход - взвешенная сумма Value ---
# Выход = attention_weights @ V
# Каждая позиция получает взвешенную комбинацию всех Value
output = attention_weights @ V  # (4,4) @ (4,3) = (4,3)
# Выводим результат
print('Выход Self-Attention (attention_weights @ V):')
print(np.round(output, 3))
# Сравниваем с исходными входами
print(f'\nИсходные входы X для сравнения:')
print(np.round(X, 3))
# Каждый выход - это "контекстуализированное" представление слова
# Слово учитло информацию из других слов через внимание


In [ ]:
# --- Шаг 4.7: Визуализация весов внимания ---
fig, ax = plt.subplots(figsize=(8, 6))
# Создаём heatmap весов внимания
im = ax.imshow(attention_weights, cmap='YlOrRd', vmin=0, vmax=1)
# Подписи осей - слова
words = ['Слово1', 'Слово2', 'Слово3', 'Слово4']
ax.set_xticks(range(4))  # Позиции по X
ax.set_yticks(range(4))  # Позиции по Y
ax.set_xticklabels(words, fontsize=11)  # Подписи X
ax.set_yticklabels(words, fontsize=11)  # Подписи Y
ax.set_xlabel('Ключи (на что смотрим)', fontsize=12)  # Ось X
ax.set_ylabel('Запросы (кто смотрит)', fontsize=12)  # Ось Y
ax.set_title('Веса внимания (Attention Weights)', fontsize=14, fontweight='bold')  # Заголовок

# Добавляем числа в каждую ячейку
for i in range(4):
    for j in range(4):
        # Текст с весом внимания
        ax.text(j, i, f'{attention_weights[i,j]:.3f}', ha='center', va='center',
                fontsize=12, color='white' if attention_weights[i,j] > 0.5 else 'black')

# Цветовая шкала
plt.colorbar(im, ax=ax, label='Вес внимания')
plt.tight_layout()
plt.show()
# Пояснение
print('Ячейка [i,j] показывает, насколько слово i "обращает внимание" на слово j')
print('Яркие клетки = сильное внимание, тёмные = слабое')


## Шаг 5. Интерактивный визуализатор внимания

Теперь вы можете сами менять Q, K, V и видеть, как меняются веса внимания.
Попробуйте разные значения и понаблюдайте за результатом!


In [ ]:
# --- Интерактивный визуализатор Self-Attention ---
def visualize_attention(q_scale=1.0, k_scale=1.0, v_scale=1.0, temperature=1.0):
    # q_scale: множитель для Query
    # k_scale: множитель для Key
    # v_scale: множитель для Value
    # temperature: температура softmax (выше = более равномерное внимание)

    # Фиксируем случайность для стабильности
    np.random.seed(42)
    # Создаём входные данные
    X = np.random.randn(4, 3)
    # Создаём матрицы весов
    W_q = np.random.randn(3, 3) * 0.5
    W_k = np.random.randn(3, 3) * 0.5
    W_v = np.random.randn(3, 3) * 0.5

    # Вычисляем Q, K, V с масштабированием
    Q = X @ W_q * q_scale  # Query с множителем
    K = X @ W_k * k_scale  # Key с множителем
    V = X @ W_v * v_scale  # Value с множителем

    # Вычисляем оценки внимания
    scores = Q @ K.T / np.sqrt(K.shape[1])  # Масштабированные оценки
    # Делим на температуру (выше = более равномерное распределение)
    scaled = scores / temperature
    # Softmax
    def softmax(x):
        e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
        return e_x / e_x.sum(axis=-1, keepdims=True)
    # Вычисляем веса внимания
    weights = softmax(scaled)
    # Вычисляем выход
    output = weights @ V

    # Визуализация
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    words = ['Слово1', 'Слово2', 'Слово3', 'Слово4']

    # Левый график: веса внимания
    ax = axes[0]
    im = ax.imshow(weights, cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels(words, fontsize=10)
    ax.set_yticklabels(words, fontsize=10)
    ax.set_xlabel('Ключи', fontsize=11)
    ax.set_ylabel('Запросы', fontsize=11)
    ax.set_title('Веса внимания', fontsize=13, fontweight='bold')
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{weights[i,j]:.2f}', ha='center', va='center',
                    fontsize=10, color='white' if weights[i,j] > 0.5 else 'black')
    plt.colorbar(im, ax=ax)

    # Правый график: выходы
    ax = axes[1]
    ax.imshow(output, cmap='coolwarm', aspect='auto')
    ax.set_xticks(range(3))
    ax.set_yticks(range(4))
    ax.set_yticklabels(words, fontsize=10)
    ax.set_xlabel('Признаки', fontsize=11)
    ax.set_ylabel('Слова', fontsize=11)
    ax.set_title('Выход Self-Attention', fontsize=13, fontweight='bold')
    for i in range(4):
        for j in range(3):
            ax.text(j, i, f'{output[i,j]:.2f}', ha='center', va='center', fontsize=10)
    plt.colorbar(ax.images[0], ax=ax)
    plt.tight_layout()
    plt.show()

    # Выводим энтропию (насколько "размазано" внимание)
    entropy = -np.sum(weights * np.log(weights + 1e-10), axis=1)  # Энтропия каждой строки
    max_entropy = np.log(4)  # Максимальная энтропия для 4 элементов
    print(f'Энтропия внимания: {np.round(entropy, 3)} (макс={max_entropy:.3f})')
    print(f'Низкая энтропия = фокус на одном элементе')
    print(f'Высокая энтропия = равномерное внимание ко всем')

# Создаём интерактивные слайдеры
q_slider = widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='Q scale')  # Масштаб Query
k_slider = widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='K scale')  # Масштаб Key
v_slider = widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='V scale')  # Масштаб Value
temp_slider = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Temperature')  # Температура

# Создаём интерактивный виджет
interactive_plot = widgets.interactive(visualize_attention,
    q_scale=q_slider, k_scale=k_slider, v_scale=v_slider, temperature=temp_slider)
# Отображаем виджет
display(interactive_plot)


## Шаг 6. Multi-Head Attention - зачем нужны несколько голов?

### Проблема одного внимания

Один механизм внимания даёт **один набор весов** для каждого слова.
Но слово может быть связано с другими словами **по-разному**:

- "Кот **который** сидел на коврике" - "который" связан с "кот" (синтаксис)
- "Кот который **сидел** на коврике" - "сидел" описывает действие (семантика)
- "Кот который сидел **на** коврике" - "на" указывает место (пространство)

### Решение: Multi-Head Attention

Используем **несколько голов внимания** параллельно.
Каждая голова "специализируется" на своём типе связей.

```
X -> [Head1, Head2, ..., HeadH] -> Concat -> Linear -> Output
     (каждая голова - свой Q,K,V)
```

Формально:
- Разбиваем Q, K, V на H частей (голов)
- Каждая голова вычисляет своё внимание
- Конкатенируем результаты
- Пропускаем через линейный слой


In [ ]:
# --- Multi-Head Attention: реализация на NumPy ---
def multi_head_attention(X, n_heads=3, d_model=6, seed=42):
    # X: входная матрица (seq_len, d_model)
    # n_heads: количество голов внимания
    # d_model: размерность модели

    np.random.seed(seed)  # Фиксируем случайность
    seq_len = X.shape[0]  # Длина последовательности
    d_k = d_model // n_heads  # Размерность каждой головы

    # Создаём матрицы весов для проекции Q, K, V
    W_q = np.random.randn(d_model, d_model) * 0.3  # (6,6) для Query
    W_k = np.random.randn(d_model, d_model) * 0.3  # (6,6) для Key
    W_v = np.random.randn(d_model, d_model) * 0.3  # (6,6) для Value
    W_o = np.random.randn(d_model, d_model) * 0.3  # (6,6) выходная проекция

    # Вычисляем Q, K, V
    Q = X @ W_q  # (seq_len, d_model)
    K = X @ W_k  # (seq_len, d_model)
    V = X @ W_v  # (seq_len, d_model)

    # Разбиваем на головы: (seq_len, d_model) -> (n_heads, seq_len, d_k)
    Q_heads = Q.reshape(seq_len, n_heads, d_k).transpose(1, 0, 2)  # (n_heads, seq_len, d_k)
    K_heads = K.reshape(seq_len, n_heads, d_k).transpose(1, 0, 2)  # (n_heads, seq_len, d_k)
    V_heads = V.reshape(seq_len, n_heads, d_k).transpose(1, 0, 2)  # (n_heads, seq_len, d_k)

    # Вычисляем внимание для каждой головы
    def softmax(x):
        e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
        return e_x / e_x.sum(axis=-1, keepdims=True)

    head_outputs = []  # Список выходов каждой головы
    head_weights = []  # Список весов каждой головы

    for h in range(n_heads):
        # Оценки внимания для головы h
        scores = Q_heads[h] @ K_heads[h].T / np.sqrt(d_k)  # (seq_len, seq_len)
        # Веса внимания
        weights = softmax(scores)  # (seq_len, seq_len)
        # Выход головы
        head_out = weights @ V_heads[h]  # (seq_len, d_k)
        # Сохраняем результаты
        head_outputs.append(head_out)
        head_weights.append(weights)

    # Конкатенируем выходы всех голов: (n_heads, seq_len, d_k) -> (seq_len, d_model)
    concat = np.concatenate(head_outputs, axis=-1)  # (seq_len, d_model)
    # Выходная проекция
    output = concat @ W_o  # (seq_len, d_model)

    return output, head_weights, head_outputs

# Тестируем Multi-Head Attention
np.random.seed(42)
# Вход: 4 слова, размерность модели 6
X_test = np.random.randn(4, 6)
# Вычисляем Multi-Head Attention с 3 головами
output, head_weights, head_outputs = multi_head_attention(X_test, n_heads=3, d_model=6)

# Выводим форму результата
print(f'Форма входа: {X_test.shape}')
print(f'Форма выхода: {output.shape}')
print(f'Количество голов: 3, d_k = 2')
print(f'Форма весов каждой головы: {head_weights[0].shape}')


In [ ]:
# --- Визуализация весов каждой головы Multi-Head Attention ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
words = ['Слово1', 'Слово2', 'Слово3', 'Слово4']

for h in range(3):
    ax = axes[h]
    # Рисуем heatmap весов для головы h
    im = ax.imshow(head_weights[h], cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels(words, fontsize=9)
    ax.set_yticklabels(words, fontsize=9)
    ax.set_title(f'Голова {h+1}', fontsize=13, fontweight='bold')
    # Добавляем числа
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{head_weights[h][i,j]:.2f}', ha='center', va='center',
                    fontsize=9, color='white' if head_weights[h][i,j] > 0.5 else 'black')

# Общий заголовок
plt.suptitle('Multi-Head Attention: веса каждой головы', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
# Пояснение
print('Каждая голова "смотрит" на разные паттерны связей!')
print('Голова 1 может улавливать синтаксис, голова 2 - семантику, и т.д.')


## Шаг 7. Позиционное кодирование (Positional Encoding)

### Проблема: Transformer не знает порядок слов!

В RNN порядок задаётся самой архитектурой (последовательная обработка).
Но Transformer обрабатывает **все слова параллельно** -> порядок потерян!

"Кот съел рыбу" и "Рыба съела кота" - для Transformer одинаковые без позиций!

### Решение: добавляем информацию о позиции

К каждому входному вектору **прибавляем** вектор позиции.
Этот вектор строится из синусов и косинусов разной частоты:

$$PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d}}\right)$$
$$PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

Разные частоты позволяют модели определять как **абсолютные**, так и **относительные** позиции.


In [ ]:
# --- Positional Encoding: реализация ---
def positional_encoding(max_len=100, d_model=64):
    # max_len: максимальная длина последовательности
    # d_model: размерность модели (должна быть чётной)

    # Создаём матрицу позиций (max_len, d_model)
    PE = np.zeros((max_len, d_model))

    # Для каждой позиции
    for pos in range(max_len):
        # Для каждой пары размерностей (2i, 2i+1)
        for i in range(d_model // 2):
            # Вычисляем знаменатель: 10000^(2i/d)
            denominator = 10000 ** (2 * i / d_model)
            # Синус для чётной размерности
            PE[pos, 2 * i] = np.sin(pos / denominator)
            # Косинус для нечётной размерности
            PE[pos, 2 * i + 1] = np.cos(pos / denominator)

    return PE

# Вычисляем позиционное кодирование
pe = positional_encoding(max_len=50, d_model=32)
# Выводим форму
print(f'Форма позиционного кодирования: {pe.shape}')
# Показываем первые 5 позиций, первые 8 размерностей
print('\nПервые 5 позиций (первые 8 размерностей):')
print(np.round(pe[:5, :8], 3))


In [ ]:
# --- Визуализация позиционного кодирования ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Левый график: линии синусов/косинусов для разных размерностей
ax = axes[0]
# Рисуем несколько размерностей
dims_to_plot = [0, 1, 4, 5, 16, 17]  # Пары (sin, cos) для разных частот
positions = np.arange(50)  # Позиции
for d in dims_to_plot:
    # Определяем тип функции
    func_type = 'sin' if d % 2 == 0 else 'cos'
    # Рисуем линию для размерности d
    ax.plot(positions, pe[:50, d], label=f'dim {d} ({func_type})')
ax.set_xlabel('Позиция', fontsize=12)
ax.set_ylabel('Значение', fontsize=12)
ax.set_title('Синусы и косинусы разных частот', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Правый график: полная heatmap позиционного кодирования
ax = axes[1]
# Рисуем heatmap всех позиций и размерностей
im = ax.imshow(pe[:30, :16].T, cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
ax.set_xlabel('Позиция', fontsize=12)
ax.set_ylabel('Размерность', fontsize=12)
ax.set_title('Positional Encoding (heatmap)', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, label='Значение')

plt.tight_layout()
plt.show()
# Пояснение
print('Низкие размерности (верх) меняются медленно - кодируют дальние позиции')
print('Высокие размерности (низ) меняются быстро - кодируют близкие позиции')


In [ ]:
# --- Демонстрация: почему позиционное кодирование работает ---
# Покажем, что модель может определить относительную позицию
# через скалярное произведение позиционных векторов

pe = positional_encoding(max_len=50, d_model=32)  # Вычисляем PE

# Матрица похожести: скалярное произведение позиционных векторов
similarity = pe @ pe.T  # (50,50)
# Нормализуем
norms = np.linalg.norm(pe, axis=1, keepdims=True)  # Нормы векторов
cosine_sim = similarity / (norms @ norms.T + 1e-8)  # Косинусная похожесть

fig, ax = plt.subplots(figsize=(8, 6))
# Визуализируем косинусную похожесть
im = ax.imshow(cosine_sim[:30, :30], cmap='RdBu', vmin=-1, vmax=1)
ax.set_xlabel('Позиция j', fontsize=12)
ax.set_ylabel('Позиция i', fontsize=12)
ax.set_title('Косинусная похожесть PE(i) и PE(j)', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, label='Косинусная похожесть')
plt.tight_layout()
plt.show()
# Пояснение
print('Диагональ = 1 (вектор похож сам на себя)')
print('Близкие позиции имеют высокую похожесть (красные)')
print('Далёкие позиции - низкую или отрицательную (синие)')
print('Это позволяет модели определять РАССТОЯНИЕ между словами!')


## Шаг 8. Transformer Block - один блок трансформера

Полный блок Transformer Encoder состоит из:

1. **Multi-Head Self-Attention** - механизм внимания
2. **Add & Norm** - остаточная связь + Layer Normalization
3. **Feed-Forward Network** - двухслойная полносвязная сеть
4. **Add & Norm** - ещё одна остаточная связь + нормализация

Схема блока:

```
Input -> Multi-Head Attention -> Add & Norm -> FFN -> Add & Norm -> Output
  |            |                    |           |          |
  +---skip-----+                    +---skip----+
```

### Остаточные связи (Residual/Skip connections)

`output = LayerNorm(x + Sublayer(x))`

Зачем: позволяют градиентам "протекать" через слои, не затухая.

### Layer Normalization

Нормализуем каждый вектор по признакам (не по батчу как BatchNorm).
Стабилизирует обучение и ускоряет сходимость.


In [ ]:
# --- Transformer Block: реализация на PyTorch ---
class TransformerBlock(nn.Module):
    # Один блок Transformer Encoder

    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        # d_model: размерность модели
        # n_heads: количество голов внимания
        # d_ff: размерность feed-forward сети
        # dropout: вероятность dropout
        super().__init__()

        # 1. Multi-Head Self-Attention
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,  # Размерность входа
            num_heads=n_heads,  # Количество голов
            dropout=dropout,    # Dropout внутри внимания
            batch_first=True    # Формат (batch, seq, dim)
        )

        # 2. Layer Normalization после внимания
        self.norm1 = nn.LayerNorm(d_model)  # Нормализация по признакам

        # 3. Feed-Forward Network (FFN)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),   # Первый линейный слой (расширение)
            nn.GELU(),                   # Активация GELU (как в оригинальном BERT)
            nn.Dropout(dropout),         # Dropout для регуляризации
            nn.Linear(d_ff, d_model),   # Второй линейный слой (сжатие)
            nn.Dropout(dropout)          # Dropout для регуляризации
        )

        # 4. Layer Normalization после FFN
        self.norm2 = nn.LayerNorm(d_model)  # Нормализация по признакам

        # 5. Dropout для остаточных связей
        self.dropout = nn.Dropout(dropout)  # Dropout

    def forward(self, x, mask=None):
        # x: (batch, seq_len, d_model)
        # mask: маска внимания (опционально)

        # --- Блок 1: Self-Attention + Add & Norm ---
        # Сохраняем остаточную связь
        residual = x  # Запоминаем вход для skip connection
        # Вычисляем Multi-Head Self-Attention
        attn_output, attn_weights = self.attention(
            x, x, x,  # Q=K=V=x (Self-Attention: все из одного источника)
            attn_mask=mask,  # Маска внимания
            need_weights=True  # Сохраняем веса для визуализации
        )
        # Добавляем dropout к выходу внимания
        attn_output = self.dropout(attn_output)
        # Остаточная связь + нормализация
        x = self.norm1(residual + attn_output)  # Add & Norm

        # --- Блок 2: FFN + Add & Norm ---
        # Сохраняем остаточную связь
        residual = x  # Запоминаем для skip connection
        # Пропускаем через FFN
        ffn_output = self.ffn(x)  # Feed-Forward Network
        # Остаточная связь + нормализация
        x = self.norm2(residual + ffn_output)  # Add & Norm

        return x, attn_weights  # Возвращаем выход и веса внимания


# Тестируем Transformer Block
d_model = 64  # Размерность модели
n_heads = 4   # Количество голов
d_ff = 256    # Размерность FFN
# Создаём блок
block = TransformerBlock(d_model, n_heads, d_ff)
# Создаём тестовый вход: батч 2, длина 10, размерность 64
x_test = torch.randn(2, 10, d_model)
# Пропускаем через блок
out, weights = block(x_test)
# Выводим форму результата
print(f'Форма входа: {x_test.shape}')
print(f'Форма выхода: {out.shape}')
print(f'Форма весов внимания: {weights.shape}')
print(f'\nБлок Transformer работает! Вход и выход одинаковой формы.')


In [ ]:
# --- Визуализация архитектуры Transformer Block ---
fig, ax = plt.subplots(figsize=(10, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('Архитектура Transformer Block', fontsize=16, fontweight='bold', pad=20)

# Цвета для компонентов
color_attn = '#FF6B6B'     # Красный - внимание
color_norm = '#4ECDC4'     # Бирюзовый - нормализация
color_ffn = '#45B7D1'      # Синий - FFN
color_skip = '#FFA07A'     # Оранжевый - skip connection
color_io = '#98D8C8'       # Зелёный - вход/выход

# Рисуем блоки (снизу вверх)
blocks_info = [
    (5, 1.0, 'Input', color_io),             # Вход
    (5, 2.2, 'Multi-Head\nAttention', color_attn),  # Внимание
    (5, 3.4, 'Add & Norm', color_norm),       # Норм 1
    (5, 4.8, 'Feed-Forward\nNetwork', color_ffn),   # FFN
    (5, 6.0, 'Add & Norm', color_norm),       # Норм 2
    (5, 7.2, 'Output', color_io),             # Выход
]

# Координаты центров блоков
y_positions = [1.0, 2.2, 3.4, 4.8, 6.0, 7.2]

for i, (cx, cy, label, color) in enumerate(blocks_info):
    # Рисуем прямоугольник
    rect = plt.Rectangle((cx - 2, cy - 0.4), 4, 0.8, fill=True,
                          color=color, alpha=0.8, edgecolor='#333', linewidth=1.5)
    ax.add_patch(rect)
    # Подпись
    ax.text(cx, cy, label, ha='center', va='center', fontsize=11, fontweight='bold', color='white')

# Стрелки между блоками
for i in range(len(y_positions) - 1):
    y_from = y_positions[i] + 0.4  # Верх текущего блока
    y_to = y_positions[i+1] - 0.4  # Низ следующего блока
    ax.annotate('', xy=(5, y_to), xytext=(5, y_from),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Skip connection 1: от Input к Add & Norm 1
ax.annotate('', xy=(7.5, 3.4), xytext=(7.5, 1.0),
            arrowprops=dict(arrowstyle='->', color=color_skip, lw=2, linestyle='dashed'))
ax.plot([7, 7.5], [1.0, 1.0], color=color_skip, lw=2, linestyle='dashed')  # Горизонталь снизу
ax.plot([7.5, 7.5], [1.0, 3.4], color=color_skip, lw=2, linestyle='dashed')  # Вертикаль
ax.plot([7.5, 7], [3.4, 3.4], color=color_skip, lw=2, linestyle='dashed')  # Горизонталь сверху
ax.text(8.2, 2.2, 'Skip\nconnection', fontsize=9, color=color_skip, ha='center')

# Skip connection 2: от Add & Norm 1 к Add & Norm 2
ax.annotate('', xy=(7.5, 6.0), xytext=(7.5, 3.4),
            arrowprops=dict(arrowstyle='->', color=color_skip, lw=2, linestyle='dashed'))
ax.plot([7, 7.5], [3.4, 3.4], color=color_skip, lw=2, linestyle='dashed')
ax.plot([7.5, 7.5], [3.4, 6.0], color=color_skip, lw=2, linestyle='dashed')
ax.plot([7.5, 7], [6.0, 6.0], color=color_skip, lw=2, linestyle='dashed')
ax.text(8.2, 4.7, 'Skip\nconnection', fontsize=9, color=color_skip, ha='center')

plt.tight_layout()
plt.show()


## Шаг 9. Полный Transformer Encoder

Полный Encoder - это стек из N блоков Transformer:

```
Input -> Token Embedding -> + Positional Encoding
      -> Block 1 -> Block 2 -> ... -> Block N -> Output
```

В оригинальной статье "Attention Is All You Need" (2017):
- N = 6 блоков
- d_model = 512
- n_heads = 8
- d_ff = 2048

Мы используем меньшие размеры для быстрого обучения в Colab.


In [ ]:
# --- Полный Transformer Encoder ---
class TransformerEncoder(nn.Module):
    # Полный Transformer Encoder из N блоков

    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_len=512, dropout=0.1):
        # vocab_size: размер словаря
        # d_model: размерность модели
        # n_heads: количество голов внимания
        # n_layers: количество блоков (слоёв)
        # d_ff: размерность feed-forward сети
        # max_len: максимальная длина последовательности
        # dropout: вероятность dropout
        super().__init__()

        # 1. Токенная эмбеддинг (превращает индексы слов в векторы)
        self.token_embedding = nn.Embedding(vocab_size, d_model)  # (vocab_size, d_model)

        # 2. Позиционное кодирование (регистрируем как буфер, не обучаем)
        self.register_buffer('pe', self._create_pe(max_len, d_model))  # (max_len, d_model)

        # 3. Dropout для эмбеддингов
        self.embedding_dropout = nn.Dropout(dropout)  # Dropout

        # 4. Стек из N Transformer блоков
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)  # Каждый блок
            for _ in range(n_layers)  # N блоков
        ])

        # Масштабирование эмбеддинга (как в оригинальной статье)
        self.d_model = d_model  # Запоминаем размерность
        self.scale = d_model ** 0.5  # Корень из d_model

    def _create_pe(self, max_len, d_model):
        # Создаём матрицу позиционного кодирования
        pe = torch.zeros(max_len, d_model)  # Нулевая матрица
        # Вектор позиций: [0, 1, 2, ..., max_len-1]
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # (max_len, 1)
        # Знаменатель: 10000^(2i/d_model)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        # Синус для чётных размерностей
        pe[:, 0::2] = torch.sin(position * div_term)  # Чётные
        # Косинус для нечётных размерностей
        pe[:, 1::2] = torch.cos(position * div_term)  # Нечётные
        # Добавляем размерность батча
        return pe.unsqueeze(0)  # (1, max_len, d_model)

    def forward(self, x, mask=None):
        # x: (batch, seq_len) - индексы токенов
        # mask: маска внимания (опционально)

        seq_len = x.size(1)  # Длина последовательности

        # 1. Токенная эмбеддинг + масштабирование
        x = self.token_embedding(x) * self.scale  # (batch, seq_len, d_model)

        # 2. Добавляем позиционное кодирование
        x = x + self.pe[:, :seq_len, :]  # (batch, seq_len, d_model)

        # 3. Dropout для эмбеддингов
        x = self.embedding_dropout(x)  # (batch, seq_len, d_model)

        # 4. Пропускаем через все Transformer блоки
        attn_weights_all = []  # Сохраняем веса внимания каждого слоя
        for layer in self.layers:
            # Пропускаем через очередной блок
            x, attn_weights = layer(x, mask)  # (batch, seq_len, d_model)
            # Сохраняем веса для визуализации
            attn_weights_all.append(attn_weights)

        return x, attn_weights_all  # Возвращаем выход и все веса внимания


# Тестируем Transformer Encoder
vocab_size = 1000  # Размер словаря
d_model = 64       # Размерность модели
n_heads = 4        # Количество голов
n_layers = 2       # Количество слоёв
d_ff = 256         # Размерность FFN
max_len = 100      # Максимальная длина

# Создаём модель
encoder = TransformerEncoder(vocab_size, d_model, n_heads, n_layers, d_ff, max_len)
# Тестовый вход: батч 2, длина 20
x_test = torch.randint(0, vocab_size, (2, 20))
# Прямой проход
out, all_weights = encoder(x_test)
# Выводим информацию
print(f'Форма входа (индексы): {x_test.shape}')
print(f'Форма выхода: {out.shape}')
print(f'Количество слоёв с весами: {len(all_weights)}')
print(f'Форма весов каждого слоя: {all_weights[0].shape}')
# Считаем параметры модели
total_params = sum(p.numel() for p in encoder.parameters())
print(f'\nОбщее количество параметров: {total_params:,}')


## Шаг 10. Классификация текста с Transformer

Теперь применим Transformer для задачи анализа тональности текста.
Мы создадим синтетический набор данных (положительные/отрицательные отзывы),
а затем обучим модель классифицировать их.

### Архитектура классификатора

```
Текст -> Токены -> Transformer Encoder -> [CLS] токен -> Linear -> Класс
```

Используем первый токен ([CLS]) как представление всего предложения.
Это подход из BERT: специальный токен накапливает информацию обо всём тексте.


In [ ]:
# --- Генерация синтетического набора данных для анализа тональности ---
# Создаём простой "словарь настроений"
positive_words = ['хорош', 'отличн', 'прекрасн', 'люблю', 'радост', 'счастлив',
                  'замечательн', 'восхитительн', 'чудесн', 'великолепн',
                  'нравит', 'приятн', 'красив', 'добр', 'светл']
negative_words = ['плох', 'ужасн', 'отвратительн', 'ненавиж', 'грустн', 'печальн',
                  'ужас', 'кошмар', 'скучн', 'разочарован',
                  'зл', 'тосклив', 'мрачн', 'страшн', 'туп']

neutral_words = ['этот', 'тот', 'мой', 'наш', 'ваш', 'есть', 'был', 'будет',
                 'мне', 'тебе', 'нам', 'вам', 'очень', 'достаточно', 'может']

# Создаём обучающие примеры
np.random.seed(42)
random.seed(42)

# Словарь: слово -> индекс
vocab = {}  # Словарь для преобразования слов в индексы
idx = 2  # 0 = padding, 1 = [CLS]
# Добавляем положительные слова в словарь
for w in positive_words:
    vocab[w] = idx  # Присваиваем индекс
    idx += 1
# Добавляем отрицательные слова в словарь
for w in negative_words:
    vocab[w] = idx  # Присваиваем индекс
    idx += 1
# Добавляем нейтральные слова в словарь
for w in neutral_words:
    vocab[w] = idx  # Присваиваем индекс
    idx += 1

vocab_size = len(vocab) + 2  # +2 для padding и [CLS]
print(f'Размер словаря: {vocab_size}')

# Генерируем примеры
train_texts = []  # Список текстов (как индексы)
train_labels = []  # Список меток (0=негатив, 1=позитив)

for _ in range(800):  # 800 обучающих примеров
    # Случайная длина предложения (5-12 слов)
    length = random.randint(5, 12)
    # Решаем тональность
    if random.random() > 0.5:
        # Позитивный пример: больше позитивных слов
        n_pos = random.randint(2, min(4, length))  # 2-4 позитивных слова
        n_neut = length - n_pos  # Остальные нейтральные
        words = random.choices(positive_words, k=n_pos) + random.choices(neutral_words, k=n_neut)
        random.shuffle(words)  # Перемешиваем
        label = 1  # Позитив
    else:
        # Негативный пример: больше негативных слов
        n_neg = random.randint(2, min(4, length))  # 2-4 негативных слова
        n_neut = length - n_neg  # Остальные нейтральные
        words = random.choices(negative_words, k=n_neg) + random.choices(neutral_words, k=n_neut)
        random.shuffle(words)  # Перемешиваем
        label = 0  # Негатив

    # Преобразуем слова в индексы: [CLS] + слова
    indices = [1] + [vocab[w] for w in words]  # [CLS] токен в начале
    train_texts.append(indices)  # Добавляем в список
    train_labels.append(label)  # Добавляем метку

# Аналогично для тестовых данных
test_texts = []
test_labels = []
for _ in range(200):  # 200 тестовых примеров
    length = random.randint(5, 12)
    if random.random() > 0.5:
        n_pos = random.randint(2, min(4, length))
        n_neut = length - n_pos
        words = random.choices(positive_words, k=n_pos) + random.choices(neutral_words, k=n_neut)
        random.shuffle(words)
        label = 1
    else:
        n_neg = random.randint(2, min(4, length))
        n_neut = length - n_neg
        words = random.choices(negative_words, k=n_neg) + random.choices(neutral_words, k=n_neut)
        random.shuffle(words)
        label = 0
    indices = [1] + [vocab[w] for w in words]
    test_texts.append(indices)
    test_labels.append(label)

print(f'Обучающих примеров: {len(train_texts)}')
print(f'Тестовых примеров: {len(test_texts)}')
print(f'Пример обучающего текста (индексы): {train_texts[0]}')
print(f'Метка: {train_labels[0]} (0=негатив, 1=позитив)')


In [ ]:
# --- Создаём Dataset и DataLoader ---
class SentimentDataset(Dataset):
    # Датасет для анализа тональности

    def __init__(self, texts, labels, max_len=20):
        # texts: список списков индексов
        # labels: список меток
        # max_len: максимальная длина (с padding)
        self.texts = texts  # Тексты
        self.labels = labels  # Метки
        self.max_len = max_len  # Максимальная длина

    def __len__(self):
        return len(self.texts)  # Размер датасета

    def __getitem__(self, idx):
        # Получаем текст и метку по индексу
        text = self.texts[idx][:self.max_len]  # Обрезаем до max_len
        label = self.labels[idx]  # Метка
        # Padding: дополняем нулями до max_len
        padding_len = self.max_len - len(text)  # Сколько нулей нужно
        text = text + [0] * padding_len  # Добавляем padding
        # Возвращаем тензоры
        return torch.tensor(text, dtype=torch.long), torch.tensor(label, dtype=torch.long)

# Создаём датасеты
train_dataset = SentimentDataset(train_texts, train_labels, max_len=20)  # Обучающий
test_dataset = SentimentDataset(test_texts, test_labels, max_len=20)  # Тестовый

# Создаём DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)  # Обучающий
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)  # Тестовый

# Проверяем один батч
for batch_x, batch_y in train_loader:
    print(f'Форма батча X: {batch_x.shape}')  # (32, 20)
    print(f'Форма батча Y: {batch_y.shape}')  # (32,)
    print(f'Пример X: {batch_x[0]}')  # Первый пример
    print(f'Пример Y: {batch_y[0]}')  # Метка первого примера
    break  # Только первый батч


In [ ]:
# --- Модель классификатора на Transformer ---
class TransformerClassifier(nn.Module):
    # Классификатор текста на основе Transformer Encoder

    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, num_classes=2, max_len=512, dropout=0.1):
        super().__init__()
        # Transformer Encoder
        self.encoder = TransformerEncoder(
            vocab_size=vocab_size,  # Размер словаря
            d_model=d_model,       # Размерность модели
            n_heads=n_heads,       # Количество голов
            n_layers=n_layers,     # Количество слоёв
            d_ff=d_ff,             # Размерность FFN
            max_len=max_len,       # Максимальная длина
            dropout=dropout        # Dropout
        )
        # Классификационная голова
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),  # Линейный слой
            nn.GELU(),                          # Активация
            nn.Dropout(dropout),                # Dropout
            nn.Linear(d_model // 2, num_classes)  # Выходной слой
        )

    def forward(self, x):
        # x: (batch, seq_len) - индексы токенов
        # Пропускаем через Transformer Encoder
        encoder_out, attn_weights = self.encoder(x)  # (batch, seq_len, d_model)
        # Берём [CLS] токен (первая позиция)
        cls_output = encoder_out[:, 0, :]  # (batch, d_model)
        # Классифицируем
        logits = self.classifier(cls_output)  # (batch, num_classes)
        return logits, attn_weights


# Создаём модель
model = TransformerClassifier(
    vocab_size=vocab_size,  # Размер словаря
    d_model=64,             # Размерность модели (маленькая для скорости)
    n_heads=4,              # 4 головы внимания
    n_layers=2,             # 2 слоя (маленькая модель)
    d_ff=256,               # Размерность FFN
    num_classes=2,          # 2 класса (позитив/негатив)
    max_len=20,             # Максимальная длина
    dropout=0.1             # Dropout
).to(device)  # Переносим на GPU если доступен

# Функция потерь и оптимизатор
criterion = nn.CrossEntropyLoss()  # Кросс-энтропия для классификации
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)  # Adam с регуляризацией

# Считаем параметры
total_params = sum(p.numel() for p in model.parameters())
print(f'Модель создана!')
print(f'Общее количество параметров: {total_params:,}')
print(f'Устройство: {device}')


In [ ]:
# --- Обучение модели Transformer ---
# Списки для хранения истории обучения
train_losses = []  # Потери на обучении
train_accs = []    # Точность на обучении
test_accs = []     # Точность на тесте

# Количество эпох
n_epochs = 15

# Цикл обучения
for epoch in range(n_epochs):
    # --- Обучение ---
    model.train()  # Режим обучения
    total_loss = 0  # Суммарная потеря
    correct = 0     # Правильные предсказания
    total = 0       # Всего примеров

    for batch_x, batch_y in train_loader:
        # Переносим данные на устройство
        batch_x = batch_x.to(device)  # Вход
        batch_y = batch_y.to(device)  # Метки

        # Прямой проход
        logits, _ = model(batch_x)  # Предсказания
        loss = criterion(logits, batch_y)  # Потеря

        # Обратный проход
        optimizer.zero_grad()  # Обнуляем градиенты
        loss.backward()        # Вычисляем градиенты
        optimizer.step()       # Обновляем веса

        # Статистика
        total_loss += loss.item() * batch_x.size(0)  # Сумма потерь
        preds = logits.argmax(dim=-1)  # Предсказанные классы
        correct += (preds == batch_y).sum().item()  # Правильные
        total += batch_x.size(0)  # Всего

    # Средние метрики за эпоху
    avg_loss = total_loss / total  # Средняя потеря
    train_acc = correct / total  # Точность на обучении
    train_losses.append(avg_loss)  # Сохраняем потерю
    train_accs.append(train_acc)  # Сохраняем точность

    # --- Тестирование ---
    model.eval()  # Режим оценки
    correct = 0   # Правильные предсказания
    total = 0     # Всего примеров
    with torch.no_grad():  # Без вычисления градиентов
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)  # Вход
            batch_y = batch_y.to(device)  # Метки
            logits, _ = model(batch_x)  # Предсказания
            preds = logits.argmax(dim=-1)  # Предсказанные классы
            correct += (preds == batch_y).sum().item()  # Правильные
            total += batch_x.size(0)  # Всего

    test_acc = correct / total  # Точность на тесте
    test_accs.append(test_acc)  # Сохраняем точность

    # Выводим прогресс каждые 3 эпохи
    if (epoch + 1) % 3 == 0 or epoch == 0:
        print(f'Эпоха {epoch+1}/{n_epochs} | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.3f} | Test Acc: {test_acc:.3f}')

print(f'\nОбучение завершено!')
print(f'Лучшая точность на тесте: {max(test_accs):.3f}')


## Шаг 11. Визуализация внимания: на что модель смотрит

Одно из главных преимуществ Transformer - **интерпретируемость**.
Мы можем визуализировать веса внимания и увидеть, какие слова
модель считает важными для принятия решения.


In [ ]:
# --- Визуализация: кривая обучения ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Левый график: Loss
ax = axes[0]
ax.plot(range(1, n_epochs+1), train_losses, 'b-o', markersize=4, label='Train Loss')
ax.set_xlabel('Эпоха', fontsize=12)
ax.set_ylabel('Потеря (Loss)', fontsize=12)
ax.set_title('Кривая обучения', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Правый график: Точность
ax = axes[1]
ax.plot(range(1, n_epochs+1), train_accs, 'b-o', markersize=4, label='Train Accuracy')
ax.plot(range(1, n_epochs+1), test_accs, 'r-s', markersize=4, label='Test Accuracy')
ax.set_xlabel('Эпоха', fontsize=12)
ax.set_ylabel('Точность', fontsize=12)
ax.set_title('Точность обучения', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f'Финальная точность на обучении: {train_accs[-1]:.3f}')
print(f'Финальная точность на тесте: {test_accs[-1]:.3f}')


In [ ]:
# --- Визуализация весов внимания модели ---
# Берём один тестовый пример и смотрим, на что обращает внимание [CLS] токен
model.eval()  # Режим оценки

# Создаём ручной пример для наглядности
idx2word = {v: k for k, v in vocab.items()}  # Обратный словарь: индекс -> слово
idx2word[0] = '[PAD]']  # Padding токен
idx2word[1] = '[CLS]']  # CLS токен

# Берём первый тестовый пример
sample_text = test_texts[0]  # Индексы слов
sample_label = test_labels[0]  # Метка

# Добавляем padding
padded = sample_text[:20] + [0] * (20 - len(sample_text[:20]))
# Создаём тензор
input_tensor = torch.tensor([padded], dtype=torch.long).to(device)

# Прямой проход с весами внимания
with torch.no_grad():
    logits, all_attn_weights = model(input_tensor)  # Предсказание и веса
    pred = logits.argmax(dim=-1).item()  # Предсказанный класс

# Определяем реальные слова (без padding)
real_len = min(len(sample_text), 20)  # Реальная длина
words = [idx2word.get(idx, f'[{idx}]') for idx in padded[:real_len]]  # Слова

# Визуализируем внимание с каждого слоя
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for layer_idx in range(len(all_attn_weights)):
    ax = axes[layer_idx]
    # Берём внимание [CLS] токена (позиция 0) ко всем остальным
    # Усредняем по всем головам
    cls_attention = all_attn_weights[layer_idx][0, :, 0, :real_len].mean(dim=0).cpu().numpy()
    # Рисуем bar chart
    bars = ax.bar(range(real_len), cls_attention, color=['#FF6B6B' if i > 0 else '#4ECDC4' for i in range(real_len)])
    ax.set_xticks(range(real_len))
    ax.set_xticklabels(words, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Вес внимания', fontsize=11)
    ax.set_title(f'Слой {layer_idx+1}: внимание [CLS] к словам', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

# Информация о примере
label_names = ['Негатив', 'Позитив']
plt.suptitle(f'Предсказание: {label_names[pred]} | Реально: {label_names[sample_label]}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# --- Визуализация: полная heatmap внимания ---
# Берём другой пример и рисуем полную матрицу внимания
sample_idx = 5  # Индекс примера
sample_text = test_texts[sample_idx]  # Индексы слов
sample_label = test_labels[sample_idx]  # Метка

# Подготавливаем
padded = sample_text[:15] + [0] * (15 - min(len(sample_text), 15))
input_tensor = torch.tensor([padded], dtype=torch.long).to(device)
real_len = min(len(sample_text), 15)
words = [idx2word.get(idx, f'[{idx}]') for idx in padded[:real_len]]

# Прямой проход
with torch.no_grad():
    logits, all_attn_weights = model(input_tensor)
    pred = logits.argmax(dim=-1).item()

# Рисуем внимание каждого слоя (усреднённое по головам)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for layer_idx in range(len(all_attn_weights)):
    ax = axes[layer_idx]
    # Усредняем внимание по головам: (n_heads, seq, seq) -> (seq, seq)
    avg_attn = all_attn_weights[layer_idx][0].mean(dim=0).cpu().numpy()
    # Обрезаем до реальной длины
    avg_attn = avg_attn[:real_len, :real_len]
    # Рисуем heatmap
    im = ax.imshow(avg_attn, cmap='YlOrRd', vmin=0)
    ax.set_xticks(range(real_len))
    ax.set_yticks(range(real_len))
    ax.set_xticklabels(words, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(words, fontsize=8)
    ax.set_title(f'Слой {layer_idx+1}', fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle(f'Полная матрица внимания | Предсказание: {label_names[pred]} | Реально: {label_names[sample_label]}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Шаг 12. Сравнение: RNN vs LSTM vs Transformer

Теперь сравним три архитектуры на одной и той же задаче классификации.
У всех моделей будет примерно одинаковое количество параметров
для честного сравнения.


In [ ]:
# --- Модель RNN для сравнения ---
class RNNClassifier(nn.Module):
    # Простой RNN классификатор

    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=2):
        super().__init__()
        # Эмбеддинг слой
        self.embedding = nn.Embedding(vocab_size, embed_dim)  # Слой эмбеддинга
        # RNN слой
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)  # Простой RNN
        # Классификатор
        self.fc = nn.Linear(hidden_dim, num_classes)  # Линейный слой

    def forward(self, x):
        # Эмбеддинг
        emb = self.embedding(x)  # (batch, seq, embed_dim)
        # RNN
        output, hidden = self.rnn(emb)  # hidden: (1, batch, hidden_dim)
        # Берём последнее скрытое состояние
        last_hidden = hidden.squeeze(0)  # (batch, hidden_dim)
        # Классификация
        logits = self.fc(last_hidden)  # (batch, num_classes)
        return logits


# --- Модель LSTM для сравнения ---
class LSTMClassifier(nn.Module):
    # LSTM классификатор

    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=2):
        super().__init__()
        # Эмбеддинг слой
        self.embedding = nn.Embedding(vocab_size, embed_dim)  # Слой эмбеддинга
        # LSTM слой
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)  # LSTM
        # Классификатор
        self.fc = nn.Linear(hidden_dim, num_classes)  # Линейный слой

    def forward(self, x):
        # Эмбеддинг
        emb = self.embedding(x)  # (batch, seq, embed_dim)
        # LSTM
        output, (hidden, cell) = self.lstm(emb)  # hidden: (1, batch, hidden_dim)
        # Берём последнее скрытое состояние
        last_hidden = hidden.squeeze(0)  # (batch, hidden_dim)
        # Классификация
        logits = self.fc(last_hidden)  # (batch, num_classes)
        return logits


# Создаём все три модели с примерно одинаковым числом параметров
embed_dim = 64   # Размерность эмбеддинга
hidden_dim = 64  # Размерность скрытого состояния

rnn_model = RNNClassifier(vocab_size, embed_dim, hidden_dim).to(device)  # RNN
lstm_model = LSTMClassifier(vocab_size, embed_dim, hidden_dim).to(device)  # LSTM
transformer_model = TransformerClassifier(
    vocab_size=vocab_size,  # Размер словаря
    d_model=64,             # Размерность модели
    n_heads=4,              # 4 головы
    n_layers=2,             # 2 слоя
    d_ff=256,               # Размерность FFN
    num_classes=2,          # 2 класса
    max_len=20,             # Максимальная длина
    dropout=0.1             # Dropout
).to(device)  # Transformer

# Считаем параметры каждой модели
rnn_params = sum(p.numel() for p in rnn_model.parameters())  # Параметры RNN
lstm_params = sum(p.numel() for p in lstm_model.parameters())  # Параметры LSTM
trans_params = sum(p.numel() for p in transformer_model.parameters())  # Параметры Transformer

print('Количество параметров моделей:')
print(f'  RNN:         {rnn_params:,}')
print(f'  LSTM:        {lstm_params:,}')
print(f'  Transformer: {trans_params:,}')


In [ ]:
# --- Функция для обучения и оценки модели ---
def train_and_evaluate(model, train_loader, test_loader, n_epochs=15, lr=1e-3):
    # Обучает модель и возвращает историю метрик
    # model: модель PyTorch
    # train_loader: обучающий DataLoader
    # test_loader: тестовый DataLoader
    # n_epochs: количество эпох
    # lr: скорость обучения

    criterion = nn.CrossEntropyLoss()  # Функция потерь
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)  # Оптимизатор

    train_losses = []  # Потери на обучении
    train_accs = []    # Точность на обучении
    test_accs = []     # Точность на тесте
    epoch_times = []   # Время каждой эпохи

    import time  # Для замера времени

    for epoch in range(n_epochs):
        start_time = time.time()  # Засекаем время

        # --- Обучение ---
        model.train()  # Режим обучения
        total_loss = 0  # Суммарная потеря
        correct = 0     # Правильные
        total = 0       # Всего

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)  # Вход
            batch_y = batch_y.to(device)  # Метки
            logits = model(batch_x)  # Предсказания
            loss = criterion(logits, batch_y)  # Потеря
            optimizer.zero_grad()  # Обнуляем градиенты
            loss.backward()        # Обратный проход
            optimizer.step()       # Обновляем веса
            total_loss += loss.item() * batch_x.size(0)  # Сумма потерь
            preds = logits.argmax(dim=-1)  # Предсказания
            correct += (preds == batch_y).sum().item()  # Правильные
            total += batch_x.size(0)  # Всего

        avg_loss = total_loss / total  # Средняя потеря
        train_acc = correct / total  # Точность на обучении
        train_losses.append(avg_loss)  # Сохраняем
        train_accs.append(train_acc)   # Сохраняем

        # --- Тестирование ---
        model.eval()  # Режим оценки
        correct = 0   # Правильные
        total = 0     # Всего
        with torch.no_grad():  # Без градиентов
            for batch_x, batch_y in test_loader:
                batch_x = batch_x.to(device)  # Вход
                batch_y = batch_y.to(device)  # Метки
                logits = model(batch_x)  # Предсказания
                preds = logits.argmax(dim=-1)  # Предсказания
                correct += (preds == batch_y).sum().item()  # Правильные
                total += batch_x.size(0)  # Всего

        test_acc = correct / total  # Точность на тесте
        test_accs.append(test_acc)  # Сохраняем

        # Время эпохи
        elapsed = time.time() - start_time  # Прошедшее время
        epoch_times.append(elapsed)  # Сохраняем

    return {
        'train_losses': train_losses,  # Потери
        'train_accs': train_accs,      # Точность обучения
        'test_accs': test_accs,        # Точность теста
        'epoch_times': epoch_times,    # Время эпох
        'total_time': sum(epoch_times)  # Общее время
    }

# Обучаем все три модели (это может занять пару минут)
print('Обучаем RNN...')
rnn_history = train_and_evaluate(rnn_model, train_loader, test_loader, n_epochs=15, lr=1e-3)
print(f'  Готово! Время: {rnn_history["total_time"]:.1f}с, Лучшая точность: {max(rnn_history["test_accs"]):.3f}')

print('\nОбучаем LSTM...')
lstm_history = train_and_evaluate(lstm_model, train_loader, test_loader, n_epochs=15, lr=1e-3)
print(f'  Готово! Время: {lstm_history["total_time"]:.1f}с, Лучшая точность: {max(lstm_history["test_accs"]):.3f}')

print('\nОбучаем Transformer...')
trans_history = train_and_evaluate(transformer_model, train_loader, test_loader, n_epochs=15, lr=1e-3)
print(f'  Готово! Время: {trans_history["total_time"]:.1f}с, Лучшая точность: {max(trans_history["test_accs"]):.3f}')


In [ ]:
# --- Визуализация сравнения RNN vs LSTM vs Transformer ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Левый верхний: Loss
ax = axes[0, 0]
ax.plot(range(1, 16), rnn_history['train_losses'], 'r-o', markersize=3, label='RNN')
ax.plot(range(1, 16), lstm_history['train_losses'], 'b-s', markersize=3, label='LSTM')
ax.plot(range(1, 16), trans_history['train_losses'], 'g-^', markersize=3, label='Transformer')
ax.set_xlabel('Эпоха', fontsize=11)
ax.set_ylabel('Потеря (Loss)', fontsize=11)
ax.set_title('Потеря на обучении', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Правый верхний: Train Accuracy
ax = axes[0, 1]
ax.plot(range(1, 16), rnn_history['train_accs'], 'r-o', markersize=3, label='RNN')
ax.plot(range(1, 16), lstm_history['train_accs'], 'b-s', markersize=3, label='LSTM')
ax.plot(range(1, 16), trans_history['train_accs'], 'g-^', markersize=3, label='Transformer')
ax.set_xlabel('Эпоха', fontsize=11)
ax.set_ylabel('Точность', fontsize=11)
ax.set_title('Точность на обучении', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Левый нижний: Test Accuracy
ax = axes[1, 0]
ax.plot(range(1, 16), rnn_history['test_accs'], 'r-o', markersize=3, label='RNN')
ax.plot(range(1, 16), lstm_history['test_accs'], 'b-s', markersize=3, label='LSTM')
ax.plot(range(1, 16), trans_history['test_accs'], 'g-^', markersize=3, label='Transformer')
ax.set_xlabel('Эпоха', fontsize=11)
ax.set_ylabel('Точность', fontsize=11)
ax.set_title('Точность на тесте', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Правый нижний: Сводная таблица (bar chart)
ax = axes[1, 1]
models = ['RNN', 'LSTM', 'Transformer']
best_accs = [max(rnn_history['test_accs']), max(lstm_history['test_accs']), max(trans_history['test_accs'])]
times = [rnn_history['total_time'], lstm_history['total_time'], trans_history['total_time']]
colors = ['#FF6B6B', '#45B7D1', '#4ECDC4']

# Два bar chart рядом
x = np.arange(len(models))
width = 0.35
bars1 = ax.bar(x - width/2, best_accs, width, label='Лучшая точность', color=colors, alpha=0.8)
ax2 = ax.twinx()
bars2 = ax2.bar(x + width/2, times, width, label='Время (с)', color=colors, alpha=0.4, hatch='//')

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylabel('Точность', fontsize=11)
ax2.set_ylabel('Время (секунды)', fontsize=11)
ax.set_title('Сравнение моделей', fontsize=12, fontweight='bold')

# Добавляем значения на бары
for bar, val in zip(bars1, best_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar, val in zip(bars2, times):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{val:.1f}с', ha='center', va='bottom', fontsize=9)

# Легенда
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9)

plt.tight_layout()
plt.show()

# Выводим итоговую таблицу
print('\n=== Итоговое сравнение ===')
print(f'{"Модель":<15} {"Параметры":<12} {"Лучш. точность":<16} {"Время":<10}')
print('-' * 55)
print(f'{"RNN":<15} {rnn_params:<12,} {max(rnn_history["test_accs"]):<16.3f} {rnn_history["total_time"]:<10.1f}')
print(f'{"LSTM":<15} {lstm_params:<12,} {max(lstm_history["test_accs"]):<16.3f} {lstm_history["total_time"]:<10.1f}')
print(f'{"Transformer":<15} {trans_params:<12,} {max(trans_history["test_accs"]):<16.3f} {trans_history["total_time"]:<10.1f}')


## Шаг 13. Практические задания

Теперь ваша очередь! Вот 5 заданий для закрепления материала.

Каждое задание постепенно усложняется. Начните с первого и двигайтесь дальше.


In [ ]:
# --- Задание 1: Реализуйте Masked Self-Attention ---
# В задаче генерации текста модель не должна "подглядывать" в будущие токены.
# Реализуйте causal mask: каждый токен может смотреть только на себя и предыдущие.

def masked_self_attention(Q, K, V):
    # Q, K, V: матрицы (seq_len, d_k)
    # Верните: output, attention_weights

    # ВАШ КОД ЗДЕСЬ:
    # Шаг 1: Вычислите scores = Q @ K^T / sqrt(d_k)
    d_k = K.shape[1]  # Размерность ключа
    scores = Q @ K.T / np.sqrt(d_k)  # Оценки внимания

    # Шаг 2: Создайте causal mask (верхнетреугольная матрица из -inf)
    seq_len = Q.shape[0]  # Длина последовательности
    mask = np.triu(np.ones((seq_len, seq_len)), k=1) * (-1e9)  # Маска: 0 ниже диагонали, -inf выше

    # Шаг 3: Примените маску к scores
    masked_scores = scores + mask  # Добавляем маску

    # Шаг 4: Примените softmax
    def softmax(x):
        e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
        return e_x / e_x.sum(axis=-1, keepdims=True)
    attention_weights = softmax(masked_scores)  # Веса внимания

    # Шаг 5: Вычислите выход
    output = attention_weights @ V  # Взвешенная сумма Value

    return output, attention_weights

# Проверка:
np.random.seed(42)
Q_test = np.random.randn(4, 3)  # 4 токена, d_k=3
K_test = np.random.randn(4, 3)
V_test = np.random.randn(4, 3)
out, weights = masked_self_attention(Q_test, K_test, V_test)
print('Веса Masked Self-Attention:')
print(np.round(weights, 3))
print('\nВерхний правый угол должен быть ~0 (маска работает!)')
print('Нижний левый - ненулевые значения (можно смотреть назад)')


In [ ]:
# --- Задание 2: Исследуйте влияние температуры ---
# Температура контролирует "остроту" распределения внимания.
# Реализуйте функцию и поэкспериментируйте с разными значениями.

def attention_with_temperature(Q, K, V, temperature=1.0):
    # temperature > 1: более равномерное внимание (меньше фокуса)
    # temperature < 1: более острое внимание (больше фокуса)
    # temperature = 0.1: почти всё внимание на один элемент

    d_k = K.shape[1]  # Размерность ключа
    # Вычисляем оценки
    scores = Q @ K.T / np.sqrt(d_k)  # Масштабированные оценки
    # Делим на температуру
    scaled = scores / temperature  # Применяем температуру

    def softmax(x):
        e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
        return e_x / e_x.sum(axis=-1, keepdims=True)
    weights = softmax(scaled)  # Веса внимания
    output = weights @ V  # Выход
    return output, weights

# Эксперимент: визуализация для разных температур
np.random.seed(42)
Q_exp = np.random.randn(4, 3)
K_exp = np.random.randn(4, 3)
V_exp = np.random.randn(4, 3)

temperatures = [0.1, 0.5, 1.0, 2.0, 5.0]  # Разные температуры
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
words = ['Слово1', 'Слово2', 'Слово3', 'Слово4']

for idx, temp in enumerate(temperatures):
    ax = axes[idx]
    _, weights = attention_with_temperature(Q_exp, K_exp, V_exp, temperature=temp)
    im = ax.imshow(weights, cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_title(f'T={temp}', fontsize=12, fontweight='bold')
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels(words, fontsize=8)
    ax.set_yticklabels(words, fontsize=8)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{weights[i,j]:.2f}', ha='center', va='center', fontsize=8,
                    color='white' if weights[i,j] > 0.5 else 'black')

plt.suptitle('Влияние температуры на распределение внимания', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('Низкая T -> фокус на одном элементе (острый пик)')
print('Высокая T -> равномерное внимание ко всем элементам')
print('\nПопробуйте изменить значения температур и понаблюдайте!')


In [ ]:
# --- Задание 3: Реализуйте Learned Positional Encoding ---
# Вместо фиксированных синусов/косинусов используем обучаемые параметры.
# Это подход из GPT и многих современных моделей.

class LearnedPositionalEncoding(nn.Module):
    # Обучаемое позиционное кодирование

    def __init__(self, max_len, d_model):
        super().__init__()
        # Создаём обучаемую матрицу позиций
        self.pe = nn.Parameter(torch.randn(max_len, d_model) * 0.02)  # Обучаемые параметры

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        seq_len = x.size(1)  # Длина последовательности
        # Добавляем позиционное кодирование
        return x + self.pe[:seq_len, :]  # (batch, seq_len, d_model)


# Сравниваем синусоидальное и обучаемое позиционное кодирование
sin_pe = positional_encoding(max_len=50, d_model=32)  # Синусоидальное PE
learned_pe = LearnedPositionalEncoding(50, 32)  # Обучаемое PE (случайная инициализация)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Синусоидальное
ax = axes[0]
ax.imshow(sin_pe[:30, :16].T, cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
ax.set_title('Синусоидальное PE (фиксированное)', fontsize=12, fontweight='bold')
ax.set_xlabel('Позиция')
ax.set_ylabel('Размерность')

# Обучаемое (до обучения)
ax = axes[1]
with torch.no_grad():
    learned_pe_data = learned_pe.pe[:30, :16].numpy()
ax.imshow(learned_pe_data.T, cmap='RdBu', aspect='auto')
ax.set_title('Обучаемое PE (до обучения)', fontsize=12, fontweight='bold')
ax.set_xlabel('Позиция')
ax.set_ylabel('Размерность')

plt.tight_layout()
plt.show()
print('Синусоидальное PE: гладкие волны, фиксированные паттерны')
print('Обучаемое PE: случайная инициализация, но МОЖЕТ обучаться под задачу')
print('\nВаша задача: интегрируйте LearnedPositionalEncoding в TransformerEncoder!')


In [ ]:
# --- Задание 4: Добавьте ещё один слой в Transformer ---
# Измените модель классификатора: увеличьте количество слоёв с 2 до 4
# и сравните результаты.

# Создаём модель с 4 слоями
model_4layers = TransformerClassifier(
    vocab_size=vocab_size,  # Размер словаря
    d_model=64,             # Размерность модели
    n_heads=4,              # 4 головы
    n_layers=4,             # 4 СЛОЯ вместо 2
    d_ff=256,               # Размерность FFN
    num_classes=2,          # 2 класса
    max_len=20,             # Максимальная длина
    dropout=0.1             # Dropout
).to(device)  # Переносим на устройство

# Считаем параметры
params_4layers = sum(p.numel() for p in model_4layers.parameters())
params_2layers = sum(p.numel() for p in model.parameters())  # Предыдущая модель с 2 слоями
print(f'Параметры модели (2 слоя): {params_2layers:,}')
print(f'Параметры модели (4 слоя): {params_4layers:,}')
print(f'Разница: {params_4layers - params_2layers:,} дополнительных параметров')

# Обучаем
print('\nОбучаем модель с 4 слоями...')
history_4layers = train_and_evaluate(model_4layers, train_loader, test_loader, n_epochs=15, lr=1e-3)
print(f'Лучшая точность (4 слоя): {max(history_4layers["test_accs"]):.3f}')
print(f'Лучшая точность (2 слоя): {max(trans_history["test_accs"]):.3f}')

# Сравнение
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, 16), trans_history['test_accs'], 'g-^', markersize=4, label='Transformer (2 слоя)')
ax.plot(range(1, 16), history_4layers['test_accs'], 'm-D', markersize=4, label='Transformer (4 слоя)')
ax.set_xlabel('Эпоха', fontsize=12)
ax.set_ylabel('Точность на тесте', fontsize=12)
ax.set_title('Сравнение: 2 слоя vs 4 слоя', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Вывод
if max(history_4layers['test_accs']) > max(trans_history['test_accs']):
    print('4 слоя лучше! Но время обучения больше.')
else:
    print('2 слоя достаточно для этой простой задачи.')
    print('Больше слоёв помогают на сложных данных.')


In [ ]:
# --- Задание 5: Визуализируйте внимание каждой головы отдельно ---
# Для примера из обученной модели нарисуйте внимание КАЖДОЙ головы
# в ОТДЕЛЬНОМ sublot. Попробуйте найти паттерны!

# Берём пример
sample_idx = 10  # Индекс примера
sample_text = test_texts[sample_idx]  # Текст
sample_label = test_labels[sample_idx]  # Метка
padded = sample_text[:15] + [0] * (15 - min(len(sample_text), 15))
input_tensor = torch.tensor([padded], dtype=torch.long).to(device)
real_len = min(len(sample_text), 15)
words = [idx2word.get(idx, f'[{idx}]') for idx in padded[:real_len]]

# Получаем внимание от модели (используем модель с 2 слоями)
model.eval()  # Режим оценки
with torch.no_grad():
    logits, all_attn_weights = model(input_tensor)  # Прямой проход

# Рисуем внимание каждой головы для первого слоя
n_heads = 4  # Количество голов
fig, axes = plt.subplots(1, n_heads, figsize=(20, 5))

for h in range(n_heads):
    ax = axes[h]
    # Берём внимание головы h из первого слоя
    head_attn = all_attn_weights[0][0, h, :real_len, :real_len].cpu().numpy()  # (seq, seq)
    # Рисуем heatmap
    im = ax.imshow(head_attn, cmap='YlOrRd', vmin=0)
    ax.set_xticks(range(real_len))
    ax.set_yticks(range(real_len))
    ax.set_xticklabels(words, rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(words, fontsize=7)
    ax.set_title(f'Голова {h+1}', fontsize=11, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046)

pred_label = label_names[logits.argmax(dim=-1).item()]
plt.suptitle(f'Внимание каждой головы (слой 1) | Предсказание: {pred_label} | Реально: {label_names[sample_label]}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Анализ
print('Посмотрите на паттерны каждой головы:')
print('- Какая голова фокусируется на [CLS]?')
print('- Какая голова связывает соседние слова?')
print('- Какая голова "смотрит" на далёкие слова?')
print('- Все головы одинаковы или каждая "специализируется"?')
print('\nПопробуйте другой sample_idx и сравните паттерны!')


## Итоги

### Что мы изучили

1. **Почему нужно Внимание** - RNN обрабатывает последовательно, Attention - параллельно
2. **Self-Attention** - Query, Key, Value: каждый элемент "смотрит" на все остальные
3. **Масштабирование** - делим на sqrt(d_k) для стабильности градиентов
4. **Multi-Head Attention** - несколько голов = несколько типов связей
5. **Positional Encoding** - синусы/косинусы добавляют информацию о порядке
6. **Transformer Block** - Attention + FFN + Residual + LayerNorm
7. **Transformer Encoder** - стек блоков, параллельная обработка
8. **Классификация текста** - [CLS] токен накапливает информацию
9. **Интерпретируемость** - визуализация весов внимания показывает, что "видит" модель
10. **Сравнение** - Transformer обычно точнее RNN/LSTM, особенно на длинных текстах

### Ключевые формулы

| Формула | Описание |
|---------|----------|
| $\text{Attention}(Q,K,V) = \text{softmax}(\frac{QK^T}{\sqrt{d_k}})V$ | Self-Attention |
| $\text{MultiHead}(Q,K,V) = \text{Concat}(head_1, ..., head_h)W^O$ | Multi-Head |
| $PE(pos, 2i) = \sin(pos / 10000^{2i/d})$ | Позиционное кодирование |
| $\text{output} = \text{LayerNorm}(x + \text{Sublayer}(x))$ | Остаточная связь |

### Что дальше?

- **Decoder** - генерация текста (GPT)
- **Encoder-Decoder** - перевод (оригинальный Transformer)
- **BERT** - предобученный Encoder для NLP задач
- **GPT** - предобученный Decoder для генерации текста
- **Vision Transformer (ViT)** - трансформер для изображений

> Помните: вы поняли механизм внимания не потому что прочитали, а потому что **реализовали его с нуля**. Это настоящий навык!
